# Energy Feature Visualization

This notebook loads already-extracted energy feature CSVs and true energy labels. It is organized as: dataset loading, handcrafted feature inspection, baseline model diagnostics, MAEST embedding experiments, repeated PCA/Ridge model selection, final baseline fitting, and isotonic calibration. It does not extract audio features.


In [4]:
from pathlib import Path
import csv
import math

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from djprojectexploration.energy_features import (
    DEFAULT_ENERGY_MODEL_FEATURE_SET,
    ENERGY_MODEL_FEATURE_SETS,
    compare_energy_model_feature_sets,
    fit_full_energy_model,
)


def project_root_from_cwd() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    return Path.cwd().expanduser().resolve()


PROJECT_ROOT = project_root_from_cwd()
ENERGY_FEATURE_DIR = PROJECT_ROOT / 'data' / 'energy_features'
FEATURE_CSVS = sorted(ENERGY_FEATURE_DIR.glob('*_energy_features.csv'))
TRACKLIST_LABEL_CSVS = [
    PROJECT_ROOT / 'music' / 'energycurvedataset' / 'energycurvedataset_tracks.csv',
]

# Edit this list if you want to compare only specific files.
# SELECTED_FEATURE_CSVS = FEATURE_CSVS
SELECTED_FEATURE_CSVS = [
    ENERGY_FEATURE_DIR / "energycurvedataset_tracks_energy_features.csv",
]
SELECTED_LABEL_CSVS = [path for path in TRACKLIST_LABEL_CSVS if path.exists()]

print(f'Project root: {PROJECT_ROOT}')
print(f'Feature CSVs ({len(SELECTED_FEATURE_CSVS)}):')
for path in SELECTED_FEATURE_CSVS:
    print(f'  - {path.relative_to(PROJECT_ROOT)}')
print(f'True label CSVs ({len(SELECTED_LABEL_CSVS)}):')
for path in SELECTED_LABEL_CSVS:
    print(f'  - {path.relative_to(PROJECT_ROOT)}')


Project root: /Users/josephdaher/Git Repositories/djprojectexploration
Feature CSVs (1):
  - data/energy_features/energycurvedataset_tracks_energy_features.csv
True label CSVs (1):
  - music/energycurvedataset/energycurvedataset_tracks.csv


## Dataset Load

These cells load the selected feature CSVs, merge in the true labels from the tracklist CSV, and summarize which datasets are active. Change `SELECTED_FEATURE_CSVS` in the setup cell when you want to switch datasets.


In [5]:
def safe_float(value) -> float:
    try:
        text = str(value).strip()
        if not text:
            return float('nan')
        number = float(text)
        return number if np.isfinite(number) else float('nan')
    except Exception:
        return float('nan')


def mix_name_from_path(path: Path) -> str:
    stem = path.stem
    for suffix in ('_energy_features', '_tracks'):
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
    return stem.replace('_', '-')


def normalized_path_key(value) -> str:
    text = str(value or '').strip()
    if not text:
        return ''
    try:
        return str(Path(text).expanduser().resolve()).lower()
    except Exception:
        return text.lower()


def basename_key(value) -> str:
    return Path(str(value or '').strip()).name.lower()


def row_path_key(row: dict) -> str:
    return normalized_path_key(row.get('filepath') or row.get('audio_path') or '')


def row_basename_key(row: dict) -> str:
    return basename_key(row.get('filepath') or row.get('audio_path') or row.get('mp3_name') or row.get('filename') or '')


def load_true_energy_labels(paths: list[Path]) -> tuple[dict[str, float], dict[str, float]]:
    labels_by_path = {}
    labels_by_base = {}
    for path in paths:
        with path.open('r', encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                energy = safe_float(row.get('energy'))
                if not np.isfinite(energy):
                    continue
                path_key = row_path_key(row)
                base_key = row_basename_key(row)
                if path_key:
                    labels_by_path[path_key] = energy
                if base_key:
                    labels_by_base[base_key] = energy
    return labels_by_path, labels_by_base


def apply_true_energy_labels(rows: list[dict], label_paths: list[Path]) -> int:
    labels_by_path, labels_by_base = load_true_energy_labels(label_paths)
    matches = 0
    for row in rows:
        energy = labels_by_path.get(row_path_key(row))
        if energy is None:
            energy = labels_by_base.get(row_basename_key(row))
        if energy is None:
            row['energy_float'] = safe_float(row.get('energy'))
            continue
        row['energy'] = f'{energy:g}'
        row['energy_float'] = energy
        row['energy_label_source'] = 'tracklist'
        matches += 1
    return matches


def load_feature_rows(paths: list[Path]) -> list[dict]:
    rows = []
    for path in paths:
        with path.open('r', encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                out = dict(row)
                out.setdefault('mix_name', mix_name_from_path(path))
                out['energy_float'] = safe_float(out.get('energy'))
                rows.append(out)
    return rows


rows = load_feature_rows(SELECTED_FEATURE_CSVS)
true_label_matches = apply_true_energy_labels(rows, SELECTED_LABEL_CSVS)
labeled_rows = [row for row in rows if np.isfinite(row['energy_float'])]
print(f'Loaded rows: {len(rows)}')
print(f'True labels matched from tracklists: {true_label_matches}')
print(f'Labeled rows available for plots/models: {len(labeled_rows)}')
if not rows:
    raise RuntimeError('No energy feature CSVs found. Run djprojectexploration-energy-features first.')


Loaded rows: 192
True labels matched from tracklists: 192
Labeled rows available for plots/models: 192


In [6]:
mixes = sorted({str(row.get('mix_name', '')) for row in rows})
summary = []
for mix in mixes:
    mix_rows = [row for row in rows if str(row.get('mix_name', '')) == mix]
    energies = np.asarray([row['energy_float'] for row in mix_rows], dtype=np.float64)
    energies = energies[np.isfinite(energies)]
    summary.append({
        'mix': mix,
        'rows': len(mix_rows),
        'labeled': int(energies.size),
        'energy_min': float(np.min(energies)) if energies.size else float('nan'),
        'energy_mean': float(np.mean(energies)) if energies.size else float('nan'),
        'energy_max': float(np.max(energies)) if energies.size else float('nan'),
    })

fig = go.Figure(data=[go.Table(
    header=dict(values=['Mix', 'Rows', 'Labeled', 'Min', 'Mean', 'Max'], fill_color='#17202a', font=dict(color='white')),
    cells=dict(values=[
        [row['mix'] for row in summary],
        [row['rows'] for row in summary],
        [row['labeled'] for row in summary],
        [f"{row['energy_min']:.2f}" if np.isfinite(row['energy_min']) else '' for row in summary],
        [f"{row['energy_mean']:.2f}" if np.isfinite(row['energy_mean']) else '' for row in summary],
        [f"{row['energy_max']:.2f}" if np.isfinite(row['energy_max']) else '' for row in summary],
    ], align='left'))
])
fig.update_layout(title='Loaded Energy Feature CSVs', height=280, margin=dict(l=20, r=20, t=60, b=20))
fig.show()


## Handcrafted Feature Diagnostics

These cells inspect the extracted full-track and peak-30-second features, compare predefined handcrafted feature sets, and show which individual features correlate with the tagged energy labels.


In [7]:
comparison = []
for feature_set in ENERGY_MODEL_FEATURE_SETS:
    result = fit_full_energy_model(rows, feature_set=feature_set)
    comparison.append({
        'feature_set': feature_set,
        'available': bool(result.available),
        'labeled_rows': len(labeled_rows),
        'retained_features': len(result.feature_names),
        'baseline_mae': result.baseline_mae,
        'oof_mae': result.oof_mae,
        'oof_r2': result.oof_r2,
        'train_mae': result.train_mae,
    })

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Feature set', 'Available', 'Labeled', 'Features', 'Baseline MAE', 'OOF MAE', 'OOF R2', 'Train MAE'],
        fill_color='#17202a',
        font=dict(color='white'),
    ),
    cells=dict(values=[
        [row['feature_set'] for row in comparison],
        [row['available'] for row in comparison],
        [row['labeled_rows'] for row in comparison],
        [row['retained_features'] for row in comparison],
        [f"{row['baseline_mae']:.3f}" if np.isfinite(row['baseline_mae']) else '' for row in comparison],
        [f"{row['oof_mae']:.3f}" if np.isfinite(row['oof_mae']) else '' for row in comparison],
        [f"{row['oof_r2']:.3f}" if np.isfinite(row['oof_r2']) else '' for row in comparison],
        [f"{row['train_mae']:.3f}" if np.isfinite(row['train_mae']) else '' for row in comparison],
    ], align='left'))
])
fig.update_layout(title='Model Feature Set Comparison', height=280, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

bar = go.Figure()
bar.add_bar(x=[row['feature_set'] for row in comparison], y=[row['oof_mae'] for row in comparison], name='OOF MAE')
bar.add_bar(x=[row['feature_set'] for row in comparison], y=[row['baseline_mae'] for row in comparison], name='Baseline MAE')
bar.update_layout(title='Lower OOF MAE Is Better', barmode='group', yaxis_title='MAE')
bar.show()


In [8]:
metadata_fields = {
    'mix_name', 'track_number', 'title', 'name', 'artists', 'artist', 'album', 'genre',
    'filename', 'mp3_name', 'filepath', 'audio_path', 'energy', 'energy_float',
    'labeled_bpm_missing', 'key', 'onset-time', 'key shift',
}

candidate_features = []
for key in sorted({key for row in rows for key in row.keys()} - metadata_fields):
    values = np.asarray([safe_float(row.get(key)) for row in labeled_rows], dtype=np.float64)
    if np.sum(np.isfinite(values)) >= 6:
        candidate_features.append(key)

correlations = []
energy = np.asarray([row['energy_float'] for row in labeled_rows], dtype=np.float64)
for feature in candidate_features:
    values = np.asarray([safe_float(row.get(feature)) for row in labeled_rows], dtype=np.float64)
    mask = np.isfinite(energy) & np.isfinite(values)
    if np.sum(mask) >= 6 and np.std(values[mask]) > 0:
        corr = float(np.corrcoef(values[mask], energy[mask])[0, 1])
        correlations.append((feature, corr, int(np.sum(mask))))

top = sorted(correlations, key=lambda item: abs(item[1]), reverse=True)[:30]
fig = go.Figure(go.Bar(
    x=[item[1] for item in top][::-1],
    y=[item[0] for item in top][::-1],
    orientation='h',
    text=[f'n={item[2]}' for item in top][::-1],
))
fig.update_layout(title='Top Feature Correlations With Tagged Energy', xaxis_title='Pearson r', height=760)
fig.show()


In [9]:
def finite_metric(value) -> str:
    return f'{value:.3f}' if np.isfinite(value) else ''


diagnostic_rows = []
for feature_set in ENERGY_MODEL_FEATURE_SETS:
    result = fit_full_energy_model(rows, feature_set=feature_set)
    actual = np.asarray([safe_float(row.get('energy')) for row in rows], dtype=np.float64)
    oof = np.asarray(result.oof_predictions, dtype=np.float64)
    fitted = np.asarray(result.predictions, dtype=np.float64)
    oof_mask = np.isfinite(actual) & np.isfinite(oof)
    fit_mask = np.isfinite(actual) & np.isfinite(fitted)
    residual = actual[oof_mask] - oof[oof_mask]
    diagnostic_rows.append({
        'feature_set': feature_set,
        'n_oof': int(np.sum(oof_mask)),
        'features': len(result.feature_names),
        'baseline_mae': result.baseline_mae,
        'oof_mae': result.oof_mae,
        'oof_rmse': float(np.sqrt(np.mean(np.square(residual)))) if residual.size else float('nan'),
        'oof_r2': result.oof_r2,
        'oof_corr': float(np.corrcoef(actual[oof_mask], oof[oof_mask])[0, 1]) if np.sum(oof_mask) >= 3 and np.std(oof[oof_mask]) > 0 else float('nan'),
        'train_mae': result.train_mae,
        'train_r2': result.train_r2,
        'best_alpha': result.best_alpha,
        'residual_mean': float(np.mean(residual)) if residual.size else float('nan'),
        'residual_std': float(np.std(residual)) if residual.size else float('nan'),
    })

diagnostic_rows = sorted(diagnostic_rows, key=lambda row: row['oof_mae'] if np.isfinite(row['oof_mae']) else float('inf'))
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Feature set', 'N', 'Features', 'OOF MAE', 'OOF RMSE', 'OOF R2', 'OOF corr', 'Baseline MAE', 'Train MAE', 'Train R2', 'Alpha', 'Residual mean', 'Residual std'],
        fill_color='#17202a',
        font=dict(color='white', size=11),
        align='left',
    ),
    cells=dict(
        values=[
            [row['feature_set'] for row in diagnostic_rows],
            [row['n_oof'] for row in diagnostic_rows],
            [row['features'] for row in diagnostic_rows],
            [finite_metric(row['oof_mae']) for row in diagnostic_rows],
            [finite_metric(row['oof_rmse']) for row in diagnostic_rows],
            [finite_metric(row['oof_r2']) for row in diagnostic_rows],
            [finite_metric(row['oof_corr']) for row in diagnostic_rows],
            [finite_metric(row['baseline_mae']) for row in diagnostic_rows],
            [finite_metric(row['train_mae']) for row in diagnostic_rows],
            [finite_metric(row['train_r2']) for row in diagnostic_rows],
            [f"{row['best_alpha']:.4g}" if np.isfinite(row['best_alpha']) else '' for row in diagnostic_rows],
            [finite_metric(row['residual_mean']) for row in diagnostic_rows],
            [finite_metric(row['residual_std']) for row in diagnostic_rows],
        ],
        align='left',
        font=dict(size=10),
    ),
)])
fig.update_layout(title='Model Diagnostics by Feature Set', height=330, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

best = diagnostic_rows[0]
print(
    f"Best by OOF MAE: {best['feature_set']} | "
    f"OOF MAE={best['oof_mae']:.3f}, RMSE={best['oof_rmse']:.3f}, "
    f"R2={best['oof_r2']:.3f}, corr={best['oof_corr']:.3f}"
)


Best by OOF MAE: full_plus_peak30_no_bpm | OOF MAE=1.029, RMSE=1.325, R2=0.666, corr=0.817


In [10]:
def feature_series(feature: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[str]]:
    x = np.asarray([safe_float(row.get(feature)) for row in labeled_rows], dtype=np.float64)
    y = np.asarray([row['energy_float'] for row in labeled_rows], dtype=np.float64)
    hover = [
        f"{row.get('mix_name', '')}<br>{row.get('track_number', '')}. {row.get('title') or row.get('name', '')}<br>{row.get('artists') or row.get('artist', '')}<br>energy={row['energy_float']:g}"
        for row in labeled_rows
    ]
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask], np.flatnonzero(mask), list(np.asarray(hover, dtype=object)[mask])


def interpolated_mean_line(x: np.ndarray, y: np.ndarray, *, bins: int = 12) -> tuple[np.ndarray, np.ndarray]:
    if x.size < 4 or float(np.nanmin(x)) == float(np.nanmax(x)):
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)
    edges = np.linspace(float(np.nanmin(x)), float(np.nanmax(x)), bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2.0
    means = np.full(bins, np.nan, dtype=np.float64)
    for i in range(bins):
        in_bin = (x >= edges[i]) & (x < edges[i + 1])
        if i == bins - 1:
            in_bin = (x >= edges[i]) & (x <= edges[i + 1])
        if np.any(in_bin):
            means[i] = float(np.mean(y[in_bin]))
    valid = np.isfinite(means)
    if np.sum(valid) < 2:
        return centers[valid], means[valid]
    return centers, np.interp(centers, centers[valid], means[valid])


def trend_line(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if x.size < 3 or np.std(x) <= 0:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)
    x_line = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    slope, intercept = np.polyfit(x, y, deg=1)
    return x_line, slope * x_line + intercept


ranked_features = [feature for feature, _corr, _n in top]
ranked_features.extend(feature for feature in candidate_features if feature not in ranked_features)
ranked_features = [feature for feature in ranked_features if feature not in {'track_number'}]
if not ranked_features:
    raise RuntimeError('No numeric feature columns are available for the interactive feature plot.')

fig = go.Figure()
buttons = []
trace_count_per_feature = 3
for feature_index, feature in enumerate(ranked_features):
    x, y, _row_idx, hover = feature_series(feature)
    visible = feature_index == 0
    mean_x, mean_y = interpolated_mean_line(x, y)
    trend_x, trend_y = trend_line(x, y)
    corr = float(np.corrcoef(x, y)[0, 1]) if x.size >= 3 and np.std(x) > 0 else float('nan')

    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='markers',
        name='tracks',
        visible=visible,
        marker=dict(color=y, colorscale='Turbo', colorbar=dict(title='Energy'), size=8, line=dict(width=0.45, color='black'), opacity=0.82),
        text=hover,
        hovertemplate='%{text}<br>' + feature + '=%{x:.3g}<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=mean_x,
        y=mean_y,
        mode='lines+markers',
        name='binned mean',
        visible=visible,
        line=dict(color='#111827', width=3),
        marker=dict(size=6),
        hovertemplate='feature bin=%{x:.3g}<br>mean energy=%{y:.2f}<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=trend_x,
        y=trend_y,
        mode='lines',
        name='linear trend',
        visible=visible,
        line=dict(color='#dc2626', width=2, dash='dash'),
        hovertemplate='trend energy=%{y:.2f}<extra></extra>',
    ))

    visible_mask = [False] * (len(ranked_features) * trace_count_per_feature)
    start = feature_index * trace_count_per_feature
    for offset in range(trace_count_per_feature):
        visible_mask[start + offset] = True
    title = f'{feature} vs tagged energy'
    if np.isfinite(corr):
        title += f' (r={corr:.3f}, n={x.size})'
    else:
        title += f' (n={x.size})'
    buttons.append(dict(
        label=feature,
        method='update',
        args=[
            {'visible': visible_mask},
            {'title.text': title, 'xaxis.title.text': feature},
        ],
    ))

initial_feature = ranked_features[0]
initial_x, initial_y, _row_idx, _hover = feature_series(initial_feature)
initial_corr = float(np.corrcoef(initial_x, initial_y)[0, 1]) if initial_x.size >= 3 and np.std(initial_x) > 0 else float('nan')
initial_title = f'{initial_feature} vs tagged energy'
if np.isfinite(initial_corr):
    initial_title += f' (r={initial_corr:.3f}, n={initial_x.size})'
else:
    initial_title += f' (n={initial_x.size})'

fig.update_layout(
    title=initial_title,
    xaxis_title=initial_feature,
    yaxis_title='Tagged energy',
    height=680,
    margin=dict(l=70, r=40, t=110, b=70),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.0,
        xanchor='left',
        y=1.13,
        yanchor='top',
        showactive=True,
    )],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
)
fig.update_yaxes(range=[0.5, 9.5], dtick=1)
fig.show()


In [11]:
FEATURE_X = 'full_rms_mean_db'
FEATURE_Y = 'peak30_rms_mean_db'

x = np.asarray([safe_float(row.get(FEATURE_X)) for row in labeled_rows], dtype=np.float64)
y = np.asarray([safe_float(row.get(FEATURE_Y)) for row in labeled_rows], dtype=np.float64)
e = np.asarray([row['energy_float'] for row in labeled_rows], dtype=np.float64)
hover = [
    f"{row.get('mix_name', '')}<br>{row.get('track_number', '')}. {row.get('title') or row.get('name', '')}<br>{row.get('artists') or row.get('artist', '')}<br>energy={row['energy_float']}"
    for row in labeled_rows
]
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(e)

fig = go.Figure(go.Scatter(
    x=x[mask],
    y=y[mask],
    mode='markers',
    marker=dict(color=e[mask], colorscale='Turbo', colorbar=dict(title='Energy'), size=9, line=dict(width=0.5, color='black')),
    text=np.asarray(hover, dtype=object)[mask],
    hovertemplate='%{text}<br>' + FEATURE_X + '=%{x:.2f}<br>' + FEATURE_Y + '=%{y:.2f}<extra></extra>',
))
fig.update_layout(title=f'{FEATURE_X} vs {FEATURE_Y}', xaxis_title=FEATURE_X, yaxis_title=FEATURE_Y)
fig.show()


## Baseline Diagnostics

These cells fit the current default handcrafted model and show out-of-fold predictions, residuals, and the largest errors. Use this section to understand where the non-embedding features succeed or fail.


In [12]:
MODEL_FEATURE_SET = DEFAULT_ENERGY_MODEL_FEATURE_SET
model_result = fit_full_energy_model(rows, feature_set=MODEL_FEATURE_SET)

actual = np.asarray([safe_float(row.get('energy')) for row in rows], dtype=np.float64)
oof = np.asarray(model_result.oof_predictions, dtype=np.float64)
fitted = np.asarray(model_result.predictions, dtype=np.float64)
mask = np.isfinite(actual) & np.isfinite(oof)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Out-of-fold prediction', 'Residuals by tagged energy'))
fig.add_trace(go.Scatter(
    x=actual[mask], y=oof[mask], mode='markers',
    marker=dict(color=actual[mask], colorscale='Turbo', size=8, line=dict(width=0.5, color='black')),
    text=[f"{rows[i].get('title') or rows[i].get('name', '')}<br>{rows[i].get('artists') or rows[i].get('artist', '')}" for i in np.flatnonzero(mask)],
    hovertemplate='%{text}<br>actual=%{x:.2f}<br>oof=%{y:.2f}<extra></extra>',
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=actual[mask], y=(actual - oof)[mask], mode='markers',
    marker=dict(color=(actual - oof)[mask], colorscale='RdBu', size=8, line=dict(width=0.5, color='black')),
    hovertemplate='actual=%{x:.2f}<br>residual=%{y:+.2f}<extra></extra>',
), row=1, col=2)
fig.add_shape(type='line', x0=1, y0=1, x1=9, y1=9, line=dict(color='black', dash='dash'), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='black', row=1, col=2)
fig.update_layout(title=f'{MODEL_FEATURE_SET}: OOF MAE={model_result.oof_mae:.3f}, train MAE={model_result.train_mae:.3f}', height=520)
fig.update_xaxes(title_text='Tagged energy', row=1, col=1)
fig.update_yaxes(title_text='OOF predicted energy', row=1, col=1)
fig.update_xaxes(title_text='Tagged energy', row=1, col=2)
fig.update_yaxes(title_text='Actual - OOF prediction', row=1, col=2)
fig.show()


In [13]:
residual_rows = []
for i, row in enumerate(rows):
    if np.isfinite(actual[i]) and np.isfinite(oof[i]):
        residual_rows.append({
            'mix': row.get('mix_name', ''),
            'track': f"{row.get('track_number', '')}. {row.get('title') or row.get('name', '')}",
            'artists': row.get('artists') or row.get('artist', ''),
            'genre': row.get('genre', ''),
            'actual': actual[i],
            'oof': oof[i],
            'residual': actual[i] - oof[i],
            'abs_residual': abs(actual[i] - oof[i]),
        })
worst = sorted(residual_rows, key=lambda row: row['abs_residual'], reverse=True)[:12]

fig = go.Figure(data=[go.Table(
    header=dict(values=['Mix', 'Track', 'Artists', 'Genre', 'Actual', 'OOF', 'Residual', '|Residual|'], fill_color='#17202a', font=dict(color='white')),
    cells=dict(values=[
        [row['mix'] for row in worst],
        [row['track'] for row in worst],
        [row['artists'] for row in worst],
        [row['genre'] for row in worst],
        [f"{row['actual']:.2f}" for row in worst],
        [f"{row['oof']:.2f}" for row in worst],
        [f"{row['residual']:+.2f}" for row in worst],
        [f"{row['abs_residual']:.2f}" for row in worst],
    ], align='left'))
])
fig.update_layout(title='Largest Out-of-Fold Errors', height=520, margin=dict(l=20, r=20, t=60, b=20))
fig.show()


## Handcrafted ElasticNet Selection

This section treats the handcrafted features as an interpretable candidate pool. ElasticNet is useful here because it can shrink redundant loudness/spectral variants toward zero while keeping correlated groups more stable than pure Lasso. The goal is feature selection and interpretation first; MAEST remains the main predictive baseline unless these features improve held-out metrics materially.


In [14]:
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HANDCRAFTED_SELECTION_EXCLUDE = {
    "energy",
    "energy_float",
    "track_number",
    "source_duration_sec",
    "peak30_start_sec",
    "peak30_end_sec",
    "full_beat_count",
    "peak30_beat_count",
    "full_analysis_duration_sec",
    "peak30_analysis_duration_sec",
}
HANDCRAFTED_SELECTION_PREFIXES = ("full_", "peak30_")
ELASTICNET_L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]
ELASTICNET_ALPHAS = np.logspace(-3, 2, 30)
ELASTICNET_CV_SPLITS = 5
ELASTICNET_CV_REPEATS = 5
ELASTICNET_INNER_SPLITS = 5

handcrafted_candidate_features = []
for key in sorted({key for row in rows for key in row.keys()}):
    if key in HANDCRAFTED_SELECTION_EXCLUDE:
        continue
    if not key.startswith(HANDCRAFTED_SELECTION_PREFIXES):
        continue
    values = np.asarray([safe_float(row.get(key)) for row in rows], dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size >= 10 and float(np.nanstd(finite)) > 0:
        handcrafted_candidate_features.append(key)

X_handcrafted_all = np.asarray(
    [[safe_float(row.get(feature)) for feature in handcrafted_candidate_features] for row in rows],
    dtype=np.float64,
)
y_handcrafted_all = np.asarray([safe_float(row.get("energy")) for row in rows], dtype=np.float64)
handcrafted_mask = np.isfinite(y_handcrafted_all) & np.any(np.isfinite(X_handcrafted_all), axis=1)
X_handcrafted_eval = X_handcrafted_all[handcrafted_mask]
y_handcrafted_eval = y_handcrafted_all[handcrafted_mask]

elasticnet_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    ElasticNetCV(
        l1_ratio=ELASTICNET_L1_RATIOS,
        alphas=ELASTICNET_ALPHAS,
        cv=min(ELASTICNET_INNER_SPLITS, len(y_handcrafted_eval)),
        max_iter=100000,
        random_state=42,
        selection="cyclic",
    ),
)
ridge_handcrafted_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=100.0),
)

outer_cv = RepeatedKFold(
    n_splits=min(ELASTICNET_CV_SPLITS, len(y_handcrafted_eval)),
    n_repeats=ELASTICNET_CV_REPEATS,
    random_state=42,
)

elasticnet_oof = np.full_like(y_handcrafted_eval, np.nan, dtype=np.float64)
ridge_oof = np.full_like(y_handcrafted_eval, np.nan, dtype=np.float64)
selection_counts = np.zeros(len(handcrafted_candidate_features), dtype=np.int64)
coef_sums = np.zeros(len(handcrafted_candidate_features), dtype=np.float64)
abs_coef_sums = np.zeros(len(handcrafted_candidate_features), dtype=np.float64)
elasticnet_fold_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_handcrafted_eval, y_handcrafted_eval), start=1):
    enet = clone(elasticnet_pipeline)
    enet.fit(X_handcrafted_eval[train_idx], y_handcrafted_eval[train_idx])
    enet_pred = enet.predict(X_handcrafted_eval[test_idx])
    elasticnet_oof[test_idx] = enet_pred

    enet_model = enet.named_steps["elasticnetcv"]
    coefs = np.asarray(enet_model.coef_, dtype=np.float64)
    selected = np.abs(coefs) > 1e-8
    selection_counts += selected.astype(np.int64)
    coef_sums += coefs
    abs_coef_sums += np.abs(coefs)

    ridge_model = clone(ridge_handcrafted_pipeline)
    ridge_model.fit(X_handcrafted_eval[train_idx], y_handcrafted_eval[train_idx])
    ridge_pred = ridge_model.predict(X_handcrafted_eval[test_idx])
    ridge_oof[test_idx] = ridge_pred

    elasticnet_fold_rows.append({
        "fold": fold_idx,
        "selected_features": int(np.sum(selected)),
        "alpha": float(enet_model.alpha_),
        "l1_ratio": float(enet_model.l1_ratio_),
        "elasticnet_mae": float(mean_absolute_error(y_handcrafted_eval[test_idx], enet_pred)),
        "ridge_mae": float(mean_absolute_error(y_handcrafted_eval[test_idx], ridge_pred)),
    })

fold_count = len(elasticnet_fold_rows)
feature_selection_rows = []
for feature, count, coef_sum, abs_sum in zip(handcrafted_candidate_features, selection_counts, coef_sums, abs_coef_sums):
    feature_selection_rows.append({
        "feature": feature,
        "selected_folds": int(count),
        "selected_pct": float(count / fold_count) if fold_count else float("nan"),
        "mean_coef": float(coef_sum / fold_count) if fold_count else float("nan"),
        "mean_abs_coef": float(abs_sum / fold_count) if fold_count else float("nan"),
    })
feature_selection_rows = sorted(
    feature_selection_rows,
    key=lambda row: (row["selected_folds"], row["mean_abs_coef"]),
    reverse=True,
)

final_elasticnet_model = clone(elasticnet_pipeline)
final_elasticnet_model.fit(X_handcrafted_eval, y_handcrafted_eval)
final_elasticnet_coefs = np.asarray(final_elasticnet_model.named_steps["elasticnetcv"].coef_, dtype=np.float64)
elasticnet_selected_features = [
    feature for feature, coef in zip(handcrafted_candidate_features, final_elasticnet_coefs)
    if abs(coef) > 1e-8
]

elasticnet_summary = [
    {
        "model": "handcrafted_ridge_reference",
        "oof_mae": float(mean_absolute_error(y_handcrafted_eval[np.isfinite(ridge_oof)], ridge_oof[np.isfinite(ridge_oof)])),
        "oof_rmse": float(np.sqrt(mean_squared_error(y_handcrafted_eval[np.isfinite(ridge_oof)], ridge_oof[np.isfinite(ridge_oof)]))),
        "oof_r2": float(r2_score(y_handcrafted_eval[np.isfinite(ridge_oof)], ridge_oof[np.isfinite(ridge_oof)])),
        "oof_corr": float(np.corrcoef(y_handcrafted_eval[np.isfinite(ridge_oof)], ridge_oof[np.isfinite(ridge_oof)])[0, 1]),
        "median_selected_features": "",
        "median_alpha": "100",
        "median_l1_ratio": "",
    },
    {
        "model": "handcrafted_elasticnet",
        "oof_mae": float(mean_absolute_error(y_handcrafted_eval[np.isfinite(elasticnet_oof)], elasticnet_oof[np.isfinite(elasticnet_oof)])),
        "oof_rmse": float(np.sqrt(mean_squared_error(y_handcrafted_eval[np.isfinite(elasticnet_oof)], elasticnet_oof[np.isfinite(elasticnet_oof)]))),
        "oof_r2": float(r2_score(y_handcrafted_eval[np.isfinite(elasticnet_oof)], elasticnet_oof[np.isfinite(elasticnet_oof)])),
        "oof_corr": float(np.corrcoef(y_handcrafted_eval[np.isfinite(elasticnet_oof)], elasticnet_oof[np.isfinite(elasticnet_oof)])[0, 1]),
        "median_selected_features": f"{np.median([row['selected_features'] for row in elasticnet_fold_rows]):.0f}",
        "median_alpha": f"{np.median([row['alpha'] for row in elasticnet_fold_rows]):.4g}",
        "median_l1_ratio": f"{np.median([row['l1_ratio'] for row in elasticnet_fold_rows]):.2g}",
    },
]

print(f"Candidate handcrafted features: {len(handcrafted_candidate_features)}")
print(f"Outer folds: {fold_count}")
print(f"Final ElasticNet selected features: {len(elasticnet_selected_features)}")
print(elasticnet_selected_features)


Candidate handcrafted features: 75
Outer folds: 25
Final ElasticNet selected features: 33
['full_band_flux_sub', 'full_bass_band_energy_ratio', 'full_beat_rms_std_db', 'full_crest_factor_db', 'full_dynamic_range_db', 'full_high_band_energy_ratio', 'full_onset_density', 'full_onset_magnitude', 'full_rms_range_db', 'full_rms_var', 'full_short_term_loudness_range_lu', 'full_spectral_centroid_hz', 'full_spectral_rolloff_85_hz', 'full_top_quartile_rms', 'peak30_band_flux_bass', 'peak30_band_flux_mid', 'peak30_band_flux_sub', 'peak30_beat_rms_std_db', 'peak30_crest_factor_db', 'peak30_downbeat_offbeat_energy_ratio', 'peak30_dynamic_range_db', 'peak30_four_bar_phrase_energy_variance', 'peak30_high_band_energy_ratio', 'peak30_low_mid_band_energy_ratio', 'peak30_mid_band_energy_ratio', 'peak30_onset_density', 'peak30_rms_iqr_db', 'peak30_rms_mean_db', 'peak30_rms_median_db', 'peak30_short_term_loudness_std_lu', 'peak30_spectral_centroid_hz', 'peak30_top_quartile_rms', 'peak30_window_rms_db']


In [15]:
fig = go.Figure(data=[go.Table(
    columnwidth=[2.4, 0.8, 0.8, 0.8, 1.0, 0.95, 0.95, 0.95],
    header=dict(
        values=["Model", "OOF MAE", "OOF RMSE", "OOF R2", "OOF corr", "Median selected", "Median alpha", "Median l1 ratio"],
        fill_color="#17202a",
        font=dict(color="white", size=11),
        align="left",
    ),
    cells=dict(
        values=[
            [row["model"] for row in elasticnet_summary],
            [f"{row['oof_mae']:.3f}" for row in elasticnet_summary],
            [f"{row['oof_rmse']:.3f}" for row in elasticnet_summary],
            [f"{row['oof_r2']:.3f}" for row in elasticnet_summary],
            [f"{row['oof_corr']:.3f}" for row in elasticnet_summary],
            [row["median_selected_features"] for row in elasticnet_summary],
            [row["median_alpha"] for row in elasticnet_summary],
            [row["median_l1_ratio"] for row in elasticnet_summary],
        ],
        font=dict(size=10),
        align="left",
    ),
)])
fig.update_layout(title="Handcrafted Ridge vs ElasticNet", width=1150, height=260, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

top_selection = feature_selection_rows[:30]
fig = go.Figure(data=[go.Table(
    columnwidth=[3.2, 0.7, 0.8, 0.9, 0.9],
    header=dict(
        values=["Feature", "Selected folds", "Selected %", "Mean coef", "Mean |coef|"],
        fill_color="#17202a",
        font=dict(color="white", size=11),
        align="left",
    ),
    cells=dict(
        values=[
            [row["feature"] for row in top_selection],
            [row["selected_folds"] for row in top_selection],
            [f"{100 * row['selected_pct']:.0f}%" for row in top_selection],
            [f"{row['mean_coef']:+.4f}" for row in top_selection],
            [f"{row['mean_abs_coef']:.4f}" for row in top_selection],
        ],
        font=dict(size=10),
        align="left",
    ),
)])
fig.update_layout(title="Most Stable ElasticNet-Selected Handcrafted Features", width=1100, height=760, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

bar_rows = list(reversed(top_selection[:20]))
fig = go.Figure(go.Bar(
    x=[row["selected_pct"] for row in bar_rows],
    y=[row["feature"] for row in bar_rows],
    orientation="h",
    marker=dict(color=[row["mean_abs_coef"] for row in bar_rows], colorscale="Viridis"),
    hovertemplate="%{y}<br>selected=%{x:.0%}<extra></extra>",
))
fig.update_xaxes(title="Selection frequency across folds", range=[0, 1])
fig.update_yaxes(title="")
fig.update_layout(title="ElasticNet Feature Selection Stability", height=620, margin=dict(l=260, r=20, t=60, b=40))
fig.show()


In [16]:
SELECTED_HANDCRAFTED_FEATURES = [
    # Spectral balance / brightness
    "full_high_band_energy_ratio",
    "full_bass_band_energy_ratio",
    "full_spectral_centroid_hz",
    "peak30_spectral_centroid_hz",
    "peak30_mid_band_energy_ratio",
    "peak30_high_band_energy_ratio",
    "full_spectral_tilt_db_per_oct",

    # Loudness / RMS / dynamics
    "peak30_rms_median_db",
    "peak30_rms_mean_db",
    "peak30_top_quartile_rms",
    "peak30_window_rms_db",
    "full_rms_var",
    "full_rms_range_db",
    "full_dynamic_range_db",
    "peak30_dynamic_range_db",
    "peak30_crest_factor_db",
    "peak30_short_term_loudness_std_lu",
    "full_short_term_loudness_range_lu",
    "full_top_quartile_rms",

    # Spectral movement / flux
    "peak30_band_flux_bass",
    "peak30_band_flux_sub",
    "peak30_band_flux_mid",

    # Onset / rhythmic activity
    "full_onset_density",
    "full_onset_magnitude",
    "peak30_onset_density",

    # Beat / phrase structure
    "peak30_four_bar_phrase_energy_variance",
    # "peak30_beat_count",
    "peak30_downbeat_offbeat_energy_ratio",
    "full_beat_rms_std_db",
]

COMPACT_SELECTED_HANDCRAFTED_FEATURES = [
    "full_high_band_energy_ratio",
    "full_bass_band_energy_ratio",
    "full_spectral_centroid_hz",
    "peak30_spectral_centroid_hz",
    "peak30_mid_band_energy_ratio",

    "peak30_rms_median_db",
    "peak30_dynamic_range_db",
    "full_dynamic_range_db",
    "peak30_crest_factor_db",

    "peak30_band_flux_bass",
    "peak30_band_flux_sub",
    "peak30_band_flux_mid",

    "full_onset_density",
    "peak30_four_bar_phrase_energy_variance",
    "peak30_downbeat_offbeat_energy_ratio",
]

## MAEST Embedding Experiments

This section loads full-track and peak-30-second MAEST embeddings, aligns them to the labeled feature rows, and compares Ridge models using handcrafted features, full MAEST PCA features, peak30 MAEST PCA features, and combined representations. PCA is fit inside each CV pipeline when evaluating models.


In [17]:
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


MAEST_PCA_COMPONENTS = 48
MAEST_EMBEDDING_DIR = PROJECT_ROOT / 'data' / 'maest_embeddings'


def selected_maest_npz_candidates(section: str = 'full') -> list[Path]:
    candidates = []
    suffix = '' if section == 'full' else f'_{section}'
    for feature_csv in SELECTED_FEATURE_CSVS:
        stem = feature_csv.stem
        if stem.endswith('_energy_features'):
            stem = stem[:-len('_energy_features')]
        candidates.append(MAEST_EMBEDDING_DIR / f'{stem}{suffix}.npz')
    candidates.append(MAEST_EMBEDDING_DIR / f'energycurvedataset_tracks{suffix}.npz')
    seen = set()
    unique = []
    for candidate in candidates:
        if candidate not in seen:
            seen.add(candidate)
            unique.append(candidate)
    return unique


def decode_np_string(value) -> str:
    return value.decode('utf-8', errors='replace') if isinstance(value, bytes) else str(value)


def scalar_np_string(data, key: str, default: str = '') -> str:
    if key not in data.files:
        return default
    value = data[key]
    if getattr(value, 'ndim', 0) == 0:
        return decode_np_string(value.item())
    if len(value):
        return decode_np_string(value[0])
    return default


def load_maest_embeddings(npz_path: Path) -> tuple[dict[str, np.ndarray], list[str], np.ndarray, dict[str, str]]:
    data = np.load(npz_path, allow_pickle=False)
    embedding_key = 'embeddings' if 'embeddings' in data.files else 'embedding'
    if embedding_key not in data.files:
        raise KeyError(f'No embeddings array found in {npz_path}. Keys: {data.files}')
    embeddings = np.asarray(data[embedding_key], dtype=np.float64)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)
    filename_key = next((key for key in ['filenames', 'filename', 'mp3_name', 'track_name'] if key in data.files), None)
    if filename_key is None:
        raise KeyError(f'No filename array found in {npz_path}. Keys: {data.files}')
    filenames = [decode_np_string(value) for value in data[filename_key]]
    lookup = {Path(filename).name.lower(): embeddings[i] for i, filename in enumerate(filenames)}
    metadata = {
        'section': scalar_np_string(data, 'maest_section', 'full'),
        'embedding_dimension': str(int(np.asarray(data['embedding_dimension']).item())) if 'embedding_dimension' in data.files else str(embeddings.shape[1]),
    }
    return lookup, filenames, embeddings, metadata


def resolve_maest_npz(section: str) -> Path:
    found = next((path for path in selected_maest_npz_candidates(section) if path.exists()), None)
    if found is not None:
        return found
    tried = [path.relative_to(PROJECT_ROOT) for path in selected_maest_npz_candidates(section)]
    command = (
        'uv run djprojectexploration-maest-playlist '
        'music/energycurvedataset/energycurvedataset_tracks.csv '
        f'--section {section}'
    )
    raise FileNotFoundError(f'No matching MAEST {section} NPZ found. Generate it with: {command}. Tried: {tried}')


maest_full_npz = resolve_maest_npz('full')
maest_peak30_npz = resolve_maest_npz('peak30')

maest_by_basename, maest_filenames, maest_embedding_matrix, maest_full_metadata = load_maest_embeddings(maest_full_npz)
maest_peak30_by_basename, maest_peak30_filenames, maest_peak30_embedding_matrix, maest_peak30_metadata = load_maest_embeddings(maest_peak30_npz)

print(f'Loaded full MAEST embeddings: {maest_full_npz.relative_to(PROJECT_ROOT)}')
print(f'Full MAEST matrix shape: {maest_embedding_matrix.shape}')
print(f'Loaded peak30 MAEST embeddings: {maest_peak30_npz.relative_to(PROJECT_ROOT)}')
print(f'Peak30 MAEST matrix shape: {maest_peak30_embedding_matrix.shape}')


Loaded full MAEST embeddings: data/maest_embeddings/energycurvedataset_tracks.npz
Full MAEST matrix shape: (192, 768)
Loaded peak30 MAEST embeddings: data/maest_embeddings/energycurvedataset_tracks_peak30.npz
Peak30 MAEST matrix shape: (192, 768)


In [18]:
def row_maest_key(row: dict) -> str:
    return row_basename_key(row)


def handcrafted_matrix(
    rows_in: list[dict],
    feature_names: list[str] | None = None,
    feature_set: str = DEFAULT_ENERGY_MODEL_FEATURE_SET,
) -> tuple[np.ndarray, list[str]]:
    if feature_names is None:
        feature_names = list(ENERGY_MODEL_FEATURE_SETS[feature_set])
    X = np.asarray(
        [[safe_float(row.get(feature)) for feature in feature_names] for row in rows_in],
        dtype=np.float64,
    )
    return X, list(feature_names)


maest_rows = []
maest_full_vectors = []
maest_peak30_vectors = []
missing_peak30 = []
for row in rows:
    energy = safe_float(row.get('energy'))
    key = row_maest_key(row)
    full_vector = maest_by_basename.get(key)
    peak30_vector = maest_peak30_by_basename.get(key)
    if full_vector is None or not np.isfinite(energy):
        continue
    if peak30_vector is None:
        missing_peak30.append(key)
        continue
    maest_rows.append(row)
    maest_full_vectors.append(full_vector)
    maest_peak30_vectors.append(peak30_vector)

if not maest_rows:
    raise RuntimeError('No labeled rows matched both full and peak30 MAEST embeddings. Check filename alignment.')
if missing_peak30:
    print(f'Skipped {len(missing_peak30)} rows with full MAEST but no peak30 MAEST match.')

X_maest = np.asarray(maest_full_vectors, dtype=np.float64)
X_maest_full = X_maest
X_maest_peak30 = np.asarray(maest_peak30_vectors, dtype=np.float64)
X_maest_full_plus_peak30 = np.hstack([X_maest_full, X_maest_peak30])
X_handcrafted, handcrafted_feature_names = handcrafted_matrix(
    maest_rows,
    feature_names=SELECTED_HANDCRAFTED_FEATURES,
)
y_maest = np.asarray([safe_float(row.get('energy')) for row in maest_rows], dtype=np.float64)
row_labels = [f"{row.get('track_number', '')}. {row.get('title') or row.get('name', '')}" for row in maest_rows]

print(f'Rows matched to full MAEST, peak30 MAEST, and labels: {len(maest_rows)}')
print(f'Handcrafted selected feature count: {len(handcrafted_feature_names)}')
print(f'Full MAEST dims: {X_maest_full.shape[1]} | Peak30 MAEST dims: {X_maest_peak30.shape[1]}')


Rows matched to full MAEST, peak30 MAEST, and labels: 192
Handcrafted selected feature count: 28
Full MAEST dims: 768 | Peak30 MAEST dims: 768


In [19]:
def evaluate_regression_pipeline(name: str, X: np.ndarray, y: np.ndarray, pipeline, *, random_state: int = 42) -> dict:
    mask = np.isfinite(y) & np.any(np.isfinite(X), axis=1)
    X_eval = X[mask]
    y_eval = y[mask]
    labels_eval = np.asarray(row_labels, dtype=object)[mask]
    if y_eval.size < 6:
        raise RuntimeError(f'{name}: need at least 6 labeled rows, got {y_eval.size}')

    alpha_grid = np.logspace(-3, 3, 25)
    n_splits = min(5, int(y_eval.size))
    alpha_scores = []
    tune_cv = RepeatedKFold(n_splits=n_splits, n_repeats=10, random_state=random_state)
    for alpha in alpha_grid:
        fold_mae = []
        for train_idx, test_idx in tune_cv.split(X_eval, y_eval):
            model = clone(pipeline)
            model.set_params(ridge__alpha=float(alpha))
            model.fit(X_eval[train_idx], y_eval[train_idx])
            fold_mae.append(mean_absolute_error(y_eval[test_idx], model.predict(X_eval[test_idx])))
        alpha_scores.append(float(np.mean(fold_mae)))
    best_alpha = float(alpha_grid[int(np.argmin(alpha_scores))])

    oof = np.full(y_eval.shape, np.nan, dtype=np.float64)
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_idx, test_idx in cv.split(X_eval, y_eval):
        model = clone(pipeline)
        model.set_params(ridge__alpha=best_alpha)
        model.fit(X_eval[train_idx], y_eval[train_idx])
        oof[test_idx] = model.predict(X_eval[test_idx])

    final_model = clone(pipeline)
    final_model.set_params(ridge__alpha=best_alpha)
    final_model.fit(X_eval, y_eval)
    fitted = final_model.predict(X_eval)
    residual = y_eval - oof
    baseline = np.full_like(y_eval, float(np.mean(y_eval)))
    return {
        'name': name,
        'n': int(y_eval.size),
        'features': int(X_eval.shape[1]),
        'best_alpha': best_alpha,
        'baseline_mae': float(mean_absolute_error(y_eval, baseline)),
        'oof_mae': float(mean_absolute_error(y_eval, oof)),
        'oof_rmse': float(np.sqrt(mean_squared_error(y_eval, oof))),
        'oof_r2': float(r2_score(y_eval, oof)),
        'oof_corr': float(np.corrcoef(y_eval, oof)[0, 1]) if np.std(oof) > 0 else float('nan'),
        'train_mae': float(mean_absolute_error(y_eval, fitted)),
        'train_r2': float(r2_score(y_eval, fitted)),
        'actual': y_eval,
        'oof': oof,
        'fitted': fitted,
        'train_residual': y_eval - fitted,
        'residual': residual,
        'labels': labels_eval,
        'model': final_model,
    }


def maest_pca_pipeline(n_components: int):
    return make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        PCA(n_components=n_components, random_state=42),
        Ridge(alpha=1.0),
    )


def two_maest_pipeline(n_components: int, full_dim: int, peak_dim: int):
    full_columns = list(range(full_dim))
    peak_columns = list(range(full_dim, full_dim + peak_dim))
    return make_pipeline(
        ColumnTransformer(
            transformers=[
                ('maest_full', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PCA(n_components=n_components, random_state=42)), full_columns),
                ('maest_peak30', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PCA(n_components=n_components, random_state=42)), peak_columns),
            ],
            remainder='drop',
        ),
        Ridge(alpha=1.0),
    )


def handcrafted_plus_two_maest_pipeline(n_components: int, handcrafted_dim: int, full_dim: int, peak_dim: int):
    handcrafted_columns = list(range(handcrafted_dim))
    full_start = handcrafted_dim
    peak_start = handcrafted_dim + full_dim
    full_columns = list(range(full_start, full_start + full_dim))
    peak_columns = list(range(peak_start, peak_start + peak_dim))
    return make_pipeline(
        ColumnTransformer(
            transformers=[
                ('handcrafted', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), handcrafted_columns),
                ('maest_full', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PCA(n_components=n_components, random_state=42)), full_columns),
                ('maest_peak30', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PCA(n_components=n_components, random_state=42)), peak_columns),
            ],
            remainder='drop',
        ),
        Ridge(alpha=1.0),
    )


maest_components = min(MAEST_PCA_COMPONENTS, X_maest_full.shape[0] - 1, X_maest_full.shape[1], X_maest_peak30.shape[1])
if maest_components < 1:
    raise RuntimeError('Not enough MAEST rows/components for PCA.')

handcrafted_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    Ridge(alpha=1.0),
)
full_maest_pipeline = maest_pca_pipeline(maest_components)
peak30_maest_pipeline = maest_pca_pipeline(maest_components)
full_plus_peak30_pipeline = two_maest_pipeline(
    maest_components,
    X_maest_full.shape[1],
    X_maest_peak30.shape[1],
)

X_handcrafted_plus_full = np.hstack([X_handcrafted, X_maest_full])
handcrafted_columns = list(range(X_handcrafted.shape[1]))
full_columns = list(range(X_handcrafted.shape[1], X_handcrafted.shape[1] + X_maest_full.shape[1]))
handcrafted_plus_full_pipeline = make_pipeline(
    ColumnTransformer(
        transformers=[
            ('handcrafted', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), handcrafted_columns),
            ('maest_full', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PCA(n_components=maest_components, random_state=42)), full_columns),
        ],
        remainder='drop',
    ),
    Ridge(alpha=1.0),
)

X_handcrafted_plus_full_peak30 = np.hstack([X_handcrafted, X_maest_full, X_maest_peak30])
handcrafted_plus_full_peak30_pipeline = handcrafted_plus_two_maest_pipeline(
    maest_components,
    X_handcrafted.shape[1],
    X_maest_full.shape[1],
    X_maest_peak30.shape[1],
)

maest_results = [
    evaluate_regression_pipeline('handcrafted_ridge', X_handcrafted, y_maest, handcrafted_pipeline),
    evaluate_regression_pipeline(f'maest_full_pca{maest_components}_ridge', X_maest_full, y_maest, full_maest_pipeline),
    evaluate_regression_pipeline(f'maest_peak30_pca{maest_components}_ridge', X_maest_peak30, y_maest, peak30_maest_pipeline),
    evaluate_regression_pipeline(f'maest_full_plus_peak30_pca{maest_components}_ridge', X_maest_full_plus_peak30, y_maest, full_plus_peak30_pipeline),
    evaluate_regression_pipeline(f'handcrafted_plus_maest_full_pca{maest_components}_ridge', X_handcrafted_plus_full, y_maest, handcrafted_plus_full_pipeline),
    evaluate_regression_pipeline(f'handcrafted_plus_maest_full_peak30_pca{maest_components}_ridge', X_handcrafted_plus_full_peak30, y_maest, handcrafted_plus_full_peak30_pipeline),
]


In [20]:
fig = go.Figure(data=[go.Table(
    columnwidth=[3.8, 0.6, 0.8, 0.8, 0.9, 0.85, 0.85, 0.8, 0.8, 0.8, 0.8],
    header=dict(
        values=['Model', 'N', 'Features', 'Alpha', 'Baseline MAE', 'OOF MAE', 'OOF RMSE', 'OOF R2', 'OOF corr', 'Train MAE', 'Train R2'],
        fill_color='#17202a',
        font=dict(color='white', size=11),
        align='left',
    ),
    cells=dict(values=[
        [row['name'] for row in maest_results],
        [row['n'] for row in maest_results],
        [row['features'] for row in maest_results],
        [f"{row['best_alpha']:.4g}" for row in maest_results],
        [f"{row['baseline_mae']:.3f}" for row in maest_results],
        [f"{row['oof_mae']:.3f}" for row in maest_results],
        [f"{row['oof_rmse']:.3f}" for row in maest_results],
        [f"{row['oof_r2']:.3f}" for row in maest_results],
        [f"{row['oof_corr']:.3f}" if np.isfinite(row['oof_corr']) else '' for row in maest_results],
        [f"{row['train_mae']:.3f}" for row in maest_results],
        [f"{row['train_r2']:.3f}" for row in maest_results],
    ], font=dict(size=10), align='left'),
)])
fig.update_layout(title='MAEST PCA + Ridge Model Comparison', width=1450, height=360, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

best_maest = sorted(maest_results, key=lambda row: row['oof_mae'])[0]
print(f"Best MAEST experiment: {best_maest['name']} | OOF MAE={best_maest['oof_mae']:.3f}, R2={best_maest['oof_r2']:.3f}")


Best MAEST experiment: handcrafted_plus_maest_full_peak30_pca48_ridge | OOF MAE=0.702, R2=0.847


In [21]:
def plot_maest_prediction_grid(results, *, prediction_key: str, residual_key: str, title: str, yaxis_title: str, hover_prediction_label: str):
    n_models = len(results)
    n_cols = min(3, n_models)
    n_rows = int(math.ceil(n_models / n_cols))
    subplot_titles = [row['name'] for row in results]
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=subplot_titles)
    for idx, result in enumerate(results):
        row_idx = idx // n_cols + 1
        col_idx = idx % n_cols + 1
        fig.add_trace(go.Scatter(
            x=result['actual'],
            y=result[prediction_key],
            mode='markers',
            marker=dict(color=result[residual_key], colorscale='RdBu', size=8, line=dict(width=0.5, color='black')),
            text=result['labels'],
            hovertemplate=f'%{{text}}<br>actual=%{{x:.2f}}<br>{hover_prediction_label}=%{{y:.2f}}<extra></extra>',
            showlegend=False,
        ), row=row_idx, col=col_idx)
        fig.add_shape(type='line', x0=1, y0=1, x1=9, y1=9, line=dict(color='black', dash='dash'), row=row_idx, col=col_idx)
        fig.update_xaxes(title_text='Tagged energy', range=[0.5, 9.5], row=row_idx, col=col_idx)
        fig.update_yaxes(title_text=yaxis_title if col_idx == 1 else None, range=[0.5, 9.5], row=row_idx, col=col_idx)
    fig.update_layout(title=title, height=430 * n_rows, margin=dict(l=50, r=20, t=90, b=50))
    fig.show()


plot_maest_prediction_grid(
    maest_results,
    prediction_key='oof',
    residual_key='residual',
    title='MAEST Experiment Out-of-Fold Predictions',
    yaxis_title='OOF prediction',
    hover_prediction_label='oof',
)


In [22]:
plot_maest_prediction_grid(
    maest_results,
    prediction_key='fitted',
    residual_key='train_residual',
    title='MAEST Experiment In-Fold Predictions',
    yaxis_title='In-fold prediction',
    hover_prediction_label='in-fold',
)


## Repeated PCA/Ridge Sweep

This section compares MAEST PCA sizes with repeated cross-validation. The table reports held-out test metrics next to train metrics so you can choose a model that performs well without relying on excessive capacity.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PCA_COMPONENT_GRID = [16, 32, 64, 96]
PCA_COMPONENT_GRID = [
    n for n in PCA_COMPONENT_GRID
    if n < X_maest_full.shape[0] and n <= min(X_maest_full.shape[1], X_maest_peak30.shape[1])
]
ALPHA_GRID = np.logspace(-2, 4, 10)
REPEATED_CV_REPEATS = 5
REPEATED_CV_SPLITS = 5
INNER_ALPHA_SPLITS = 3

# Edit this list to control the repeated-CV sweep.
# Available variants: "full", "peak30", "full_plus_peak30",
# "handcrafted_plus_full", "handcrafted_plus_peak30", "handcrafted_plus_full_peak30".
MAEST_REPEATED_CV_VARIANTS = [
    "full",
    "peak30",
    "full_plus_peak30",
    "handcrafted_plus_full_peak30",
    "handcrafted"
]


def matrix_for_features(rows_in, feature_names):
    return np.asarray(
        [[safe_float(row.get(feature)) for feature in feature_names] for row in rows_in],
        dtype=np.float64,
    )

X_small_handcrafted = matrix_for_features(maest_rows, SELECTED_HANDCRAFTED_FEATURES)


def repeated_cv_ridge_metrics(name, X, y, pipeline, *, repeats=REPEATED_CV_REPEATS, splits=REPEATED_CV_SPLITS, random_state=42):
    mask = np.isfinite(y) & np.any(np.isfinite(X), axis=1)
    X_eval = X[mask]
    y_eval = y[mask]

    cv = RepeatedKFold(
        n_splits=min(splits, len(y_eval)),
        n_repeats=repeats,
        random_state=random_state,
    )

    rows_out = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_eval, y_eval), start=1):
        best_alpha = None
        best_mae = float("inf")

        inner_cv = KFold(
            n_splits=min(INNER_ALPHA_SPLITS, len(train_idx)),
            shuffle=True,
            random_state=random_state + fold_idx,
        )

        for alpha in ALPHA_GRID:
            fold_maes = []
            for inner_train_rel, inner_test_rel in inner_cv.split(X_eval[train_idx], y_eval[train_idx]):
                inner_train_idx = train_idx[inner_train_rel]
                inner_test_idx = train_idx[inner_test_rel]

                model = clone(pipeline)
                model.set_params(ridge__alpha=float(alpha))
                model.fit(X_eval[inner_train_idx], y_eval[inner_train_idx])
                pred = model.predict(X_eval[inner_test_idx])
                fold_maes.append(mean_absolute_error(y_eval[inner_test_idx], pred))

            alpha_mae = float(np.mean(fold_maes))
            if alpha_mae < best_mae:
                best_mae = alpha_mae
                best_alpha = float(alpha)

        model = clone(pipeline)
        model.set_params(ridge__alpha=best_alpha)
        model.fit(X_eval[train_idx], y_eval[train_idx])
        test_pred = model.predict(X_eval[test_idx])
        train_pred = model.predict(X_eval[train_idx])

        rows_out.append({
            "model": name,
            "fold": fold_idx,
            "alpha": best_alpha,
            "mae": float(mean_absolute_error(y_eval[test_idx], test_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_eval[test_idx], test_pred))),
            "r2": float(r2_score(y_eval[test_idx], test_pred)),
            "corr": float(np.corrcoef(y_eval[test_idx], test_pred)[0, 1]) if np.std(test_pred) > 0 else float("nan"),
            "train_mae": float(mean_absolute_error(y_eval[train_idx], train_pred)),
            "train_rmse": float(np.sqrt(mean_squared_error(y_eval[train_idx], train_pred))),
            "train_r2": float(r2_score(y_eval[train_idx], train_pred)),
            "train_corr": float(np.corrcoef(y_eval[train_idx], train_pred)[0, 1]) if np.std(train_pred) > 0 else float("nan"),
        })

    return rows_out


def summarize_cv(rows_in):
    models = sorted({row["model"] for row in rows_in})
    summary = []
    metrics = ["mae", "rmse", "r2", "corr", "train_mae", "train_rmse", "train_r2", "train_corr"]
    for model in models:
        model_rows = [row for row in rows_in if row["model"] == model]
        out = {
            "model": model,
            "folds": len(model_rows),
            "alpha_median": float(np.median([row["alpha"] for row in model_rows])),
        }
        for metric in metrics:
            values = np.asarray([row[metric] for row in model_rows], dtype=np.float64)
            values = values[np.isfinite(values)]
            out[f"{metric}_mean"] = float(np.mean(values)) if values.size else float("nan")
            out[f"{metric}_std"] = float(np.std(values)) if values.size else float("nan")
        out["mae_gap_mean"] = out["mae_mean"] - out["train_mae_mean"]
        out["r2_gap_mean"] = out["train_r2_mean"] - out["r2_mean"]
        summary.append(out)
    return sorted(summary, key=lambda row: row["mae_mean"])


def pca_branch(columns, n_components, name):
    return (
        name,
        make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            PCA(n_components=n_components, random_state=42),
        ),
        columns,
    )


def scaled_branch(columns, name):
    return (
        name,
        make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
        columns,
    )


def make_multi_view_ridge_pipeline(views, n_components):
    matrices = [matrix for _, matrix in views]
    X = np.hstack(matrices)
    transformers = []
    cursor = 0
    for view_name, matrix in views:
        columns = list(range(cursor, cursor + matrix.shape[1]))
        cursor += matrix.shape[1]
        if view_name == "handcrafted":
            transformers.append(scaled_branch(columns, view_name))
        else:
            transformers.append(pca_branch(columns, n_components, view_name))
    pipeline = make_pipeline(
        ColumnTransformer(transformers=transformers, remainder="drop"),
        Ridge(alpha=1.0),
    )
    return X, pipeline


def maest_variant_views(variant: str):
    if variant == "full":
        return [("maest_full", X_maest_full)]
    if variant == "peak30":
        return [("maest_peak30", X_maest_peak30)]
    if variant == "full_plus_peak30":
        return [("maest_full", X_maest_full), ("maest_peak30", X_maest_peak30)]
    if variant == "handcrafted_plus_full":
        return [("handcrafted", X_small_handcrafted), ("maest_full", X_maest_full)]
    if variant == "handcrafted_plus_peak30":
        return [("handcrafted", X_small_handcrafted), ("maest_peak30", X_maest_peak30)]
    if variant == "handcrafted_plus_full_peak30":
        return [("handcrafted", X_small_handcrafted), ("maest_full", X_maest_full), ("maest_peak30", X_maest_peak30)]
    raise ValueError(f"Unknown MAEST repeated-CV variant: {variant}")


def model_name_for_variant(variant: str, n_components: int) -> str:
    if variant == "full":
        return f"maest_full_pca{n_components}"
    if variant == "peak30":
        return f"maest_peak30_pca{n_components}"
    if variant == "full_plus_peak30":
        return f"maest_full_plus_peak30_pca{n_components}"
    if variant == "handcrafted_plus_full":
        return f"small_handcrafted_plus_maest_full_pca{n_components}"
    if variant == "handcrafted_plus_peak30":
        return f"small_handcrafted_plus_maest_peak30_pca{n_components}"
    if variant == "handcrafted_plus_full_peak30":
        return f"small_handcrafted_plus_maest_full_peak30_pca{n_components}"
    raise ValueError(f"Unknown MAEST repeated-CV variant: {variant}")


all_cv_rows = []

for n_components in PCA_COMPONENT_GRID:
    for variant in MAEST_REPEATED_CV_VARIANTS:
        X_variant, variant_pipeline = make_multi_view_ridge_pipeline(
            maest_variant_views(variant),
            n_components,
        )
        all_cv_rows.extend(
            repeated_cv_ridge_metrics(
                model_name_for_variant(variant, n_components),
                X_variant,
                y_maest,
                variant_pipeline,
            )
        )

cv_summary = summarize_cv(all_cv_rows)
print(f"Compared {len(cv_summary)} models across {len(all_cv_rows)} folds.")
print(f"Variants: {MAEST_REPEATED_CV_VARIANTS}")
print(f"PCA components: {PCA_COMPONENT_GRID}")
print(f"Alpha grid: {[float(a) for a in ALPHA_GRID]}")


Compared 16 models across 400 folds.
Variants: ['full', 'peak30', 'full_plus_peak30', 'handcrafted_plus_full_peak30']
PCA components: [16, 32, 64, 96]
Alpha grid: [0.01, 0.046415888336127774, 0.21544346900318834, 1.0, 4.6415888336127775, 21.54434690031882, 100.0, 464.1588833612773, 2154.4346900318824, 10000.0]


In [24]:
fig = go.Figure(data=[go.Table(
    columnwidth=[3.8, 0.55, 0.85, 0.85, 0.85, 0.85, 0.85, 0.85, 0.85, 0.85, 0.75, 0.75, 0.9],
    header=dict(
        values=[
            "Model", "Folds", "Test MAE", "Test MAE std", "Train MAE", "Train MAE std",
            "MAE gap", "Test R2", "Test R2 std", "Train R2", "R2 gap", "Corr", "Median alpha"
        ],
        fill_color="#17202a",
        font=dict(color="white", size=11),
        align="left",
    ),
    cells=dict(
        values=[
            [row["model"] for row in cv_summary],
            [row["folds"] for row in cv_summary],
            [f"{row['mae_mean']:.3f}" for row in cv_summary],
            [f"{row['mae_std']:.3f}" for row in cv_summary],
            [f"{row['train_mae_mean']:.3f}" for row in cv_summary],
            [f"{row['train_mae_std']:.3f}" for row in cv_summary],
            [f"{row['mae_gap_mean']:.3f}" for row in cv_summary],
            [f"{row['r2_mean']:.3f}" for row in cv_summary],
            [f"{row['r2_std']:.3f}" for row in cv_summary],
            [f"{row['train_r2_mean']:.3f}" for row in cv_summary],
            [f"{row['r2_gap_mean']:.3f}" for row in cv_summary],
            [f"{row['corr_mean']:.3f}" for row in cv_summary],
            [f"{row['alpha_median']:.4g}" for row in cv_summary],
        ],
        font=dict(size=10),
        align="left",
    ),
)])

fig.update_layout(
    title="Repeated-CV MAEST PCA Sweep: Test vs Train",
    width=1450,
    height=620,
    margin=dict(l=20, r=20, t=60, b=20),
)
fig.show()

best = cv_summary[0]
print(
    f"Best repeated-CV model: {best['model']} | "
    f"test MAE={best['mae_mean']:.3f} ± {best['mae_std']:.3f}, "
    f"train MAE={best['train_mae_mean']:.3f}, "
    f"MAE gap={best['mae_gap_mean']:.3f}, "
    f"test R2={best['r2_mean']:.3f} ± {best['r2_std']:.3f}, "
    f"train R2={best['train_r2_mean']:.3f}"
)


Best repeated-CV model: small_handcrafted_plus_maest_full_peak30_pca64 | test MAE=0.700 ± 0.085, train MAE=0.406, MAE gap=0.294, test R2=0.845 ± 0.037, train R2=0.949


In [35]:
import pandas as pd
import re

PAPER_MODEL_ORDER = [
    "full",
    "peak30",
    "full_plus_peak30",
    "handcrafted_plus_full_peak30",
]

PAPER_MODEL_LABELS = {
    "full": "MAEST full",
    "peak30": "MAEST peak30",
    "full_plus_peak30": "MAEST full + peak30",
    "handcrafted_plus_full_peak30": "Handcrafted + MAEST full + peak30",
}

def parse_model_family(model_name: str) -> str | None:
    name = str(model_name)

    if name.startswith("small_handcrafted_plus_maest_full_peak30_pca64"):
        return "handcrafted_plus_full_peak30"
    if name.startswith("handcrafted_plus_maest_full_peak30_pca64"):
        return "handcrafted_plus_full_peak30"
    if name.startswith("maest_full_plus_peak30_pca64"):
        return "full_plus_peak30"
    if name.startswith("maest_peak30_pca64"):
        return "peak30"
    if name.startswith("maest_full_pca64"):
        return "full"

    return None

def parse_pca_components(model_name: str):
    match = re.search(r"_pca(\d+)", str(model_name))
    return int(match.group(1)) if match else None

paper_candidates = []
for row in cv_summary:
    family = parse_model_family(row["model"])
    if family is None:
        continue

    out = dict(row)
    out["family"] = family
    out["label"] = PAPER_MODEL_LABELS[family]
    out["pca_components"] = parse_pca_components(row["model"])
    paper_candidates.append(out)

# Pick best PCA setting within each model family by mean test MAE.
paper_rows = []
for family in PAPER_MODEL_ORDER:
    family_rows = [r for r in paper_candidates if r["family"] == family]
    if not family_rows:
        continue
    best_row = min(family_rows, key=lambda r: r["mae_mean"])
    paper_rows.append(best_row)

paper_df = pd.DataFrame([
    {
        "Model": row["label"],
        "PCA/view": "--" if row["pca_components"] is None else row["pca_components"],
        "CV folds": row["folds"],
        "MAE": row["mae_mean"],
        "MAE std": row["mae_std"],
        "RMSE": row.get("rmse_mean", float("nan")),
        "R2": row["r2_mean"],
        "R2 std": row["r2_std"],
        "Corr": row["corr_mean"],
        "Train MAE": row["train_mae_mean"],
        "MAE gap": row["mae_gap_mean"],
        "Median alpha": row["alpha_median"],
        "Raw model name": row["model"],
    }
    for row in paper_rows
])

display(paper_df)

,Model,PCA/view,CV folds,MAE,MAE std,RMSE,R2,R2 std,Corr,Train MAE,MAE gap,Median alpha,Raw model name
0,MAEST full,64,25,0.740222,0.077768,0.935509,0.826930,0.036279,0.917068,0.509947,0.230275,464.158883,maest_full_pca64
1,MAEST peak30,64,25,0.735136,0.104429,0.922285,0.831660,0.039232,0.918874,0.479260,0.255876,100.000000,maest_peak30_pca64
2,MAEST full + peak30,64,25,0.701818,0.086819,0.885721,0.844656,0.036192,0.925387,0.412831,0.288987,464.158883,maest_full_plus_peak30_pca64
3,Handcrafted + MAEST full + peak30,64,25,0.700154,0.085133,0.885357,0.844690,0.036762,0.925415,0.405997,0.294157,464.158883,small_handcrafted_plus_maest_full_peak30_pca64


## Final Full+Peak30 PCA64 Ridge Baseline

After using repeated cross-validation to choose the modeling recipe, this cell fits the selected production-style baseline on all labeled rows: full-track MAEST plus peak30 MAEST, separate PCA64 branches for each view, and Ridge. The train metrics here are not a generalization estimate; they are a quick check of the final model fit that you would use for new unlabeled tracks.


In [25]:
from sklearn.base import clone

FINAL_MAEST_PCA_COMPONENTS = 64
FINAL_ALPHA_GRID = np.logspace(-2, 4, 50)
FINAL_ALPHA_CV_SPLITS = 5

final_mask = (
    np.isfinite(y_maest)
    & np.any(np.isfinite(X_maest_full), axis=1)
    & np.any(np.isfinite(X_maest_peak30), axis=1)
)
X_final = np.hstack([X_maest_full, X_maest_peak30])[final_mask]
y_final = y_maest[final_mask]
final_labels = np.asarray(row_labels, dtype=object)[final_mask]

final_full_dim = X_maest_full.shape[1]
final_peak30_dim = X_maest_peak30.shape[1]
final_full_columns = list(range(final_full_dim))
final_peak30_columns = list(range(final_full_dim, final_full_dim + final_peak30_dim))

final_base_pipeline = make_pipeline(
    ColumnTransformer(
        transformers=[
            (
                "maest_full",
                make_pipeline(
                    SimpleImputer(strategy="median"),
                    StandardScaler(),
                    PCA(n_components=FINAL_MAEST_PCA_COMPONENTS, random_state=42),
                ),
                final_full_columns,
            ),
            (
                "maest_peak30",
                make_pipeline(
                    SimpleImputer(strategy="median"),
                    StandardScaler(),
                    PCA(n_components=FINAL_MAEST_PCA_COMPONENTS, random_state=42),
                ),
                final_peak30_columns,
            ),
        ],
        remainder="drop",
    ),
    Ridge(alpha=1.0),
)


def select_alpha_by_cv(X, y, pipeline, alpha_grid, *, splits=FINAL_ALPHA_CV_SPLITS, random_state=42):
    cv = KFold(n_splits=min(splits, len(y)), shuffle=True, random_state=random_state)
    alpha_rows = []
    for alpha in alpha_grid:
        fold_maes = []
        for train_idx, test_idx in cv.split(X, y):
            model = clone(pipeline)
            model.set_params(ridge__alpha=float(alpha))
            model.fit(X[train_idx], y[train_idx])
            pred = model.predict(X[test_idx])
            fold_maes.append(mean_absolute_error(y[test_idx], pred))
        alpha_rows.append({"alpha": float(alpha), "mae": float(np.mean(fold_maes))})
    return min(alpha_rows, key=lambda row: row["mae"]), alpha_rows


final_alpha_result, final_alpha_rows = select_alpha_by_cv(
    X_final,
    y_final,
    final_base_pipeline,
    FINAL_ALPHA_GRID,
)
FINAL_RIDGE_ALPHA = final_alpha_result["alpha"]

final_maest_full_peak30_pca64_ridge_model = clone(final_base_pipeline)
final_maest_full_peak30_pca64_ridge_model.set_params(ridge__alpha=FINAL_RIDGE_ALPHA)
final_maest_full_peak30_pca64_ridge_model.fit(X_final, y_final)
final_train_pred = final_maest_full_peak30_pca64_ridge_model.predict(X_final)
final_train_residual = y_final - final_train_pred

# Backward-compatible alias for downstream cells that used the old variable name.
final_maest_pca64_ridge_model = final_maest_full_peak30_pca64_ridge_model


def residual_skew(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return float("nan")
    centered = values - float(np.mean(values))
    std = float(np.std(centered))
    if std <= 0:
        return float("nan")
    return float(np.mean((centered / std) ** 3))


final_baseline_summary = {
    "model": f"maest_full_plus_peak30_pca{FINAL_MAEST_PCA_COMPONENTS}_ridge_final_fit",
    "n": int(len(y_final)),
    "pca_components_per_view": int(FINAL_MAEST_PCA_COMPONENTS),
    "input_features": int(X_final.shape[1]),
    "alpha": float(FINAL_RIDGE_ALPHA),
    "alpha_cv_mae": float(final_alpha_result["mae"]),
    "train_mae": float(mean_absolute_error(y_final, final_train_pred)),
    "train_rmse": float(np.sqrt(mean_squared_error(y_final, final_train_pred))),
    "train_r2": float(r2_score(y_final, final_train_pred)),
    "train_corr": float(np.corrcoef(y_final, final_train_pred)[0, 1]),
    "residual_skew": residual_skew(final_train_residual),
}

fig = go.Figure(data=[go.Table(
    columnwidth=[3.5, 0.55, 0.75, 0.75, 0.85, 1.0, 0.85, 0.85, 0.75, 0.75, 0.9],
    header=dict(
        values=["Model", "N", "Input dims", "PCA/view", "Alpha", "Alpha CV MAE", "Train MAE", "Train RMSE", "Train R2", "Train corr", "Residual skew"],
        fill_color="#17202a",
        font=dict(color="white", size=11),
        align="left",
    ),
    cells=dict(
        values=[
            [final_baseline_summary["model"]],
            [final_baseline_summary["n"]],
            [final_baseline_summary["input_features"]],
            [final_baseline_summary["pca_components_per_view"]],
            [f"{final_baseline_summary['alpha']:.4g}"],
            [f"{final_baseline_summary['alpha_cv_mae']:.3f}"],
            [f"{final_baseline_summary['train_mae']:.3f}"],
            [f"{final_baseline_summary['train_rmse']:.3f}"],
            [f"{final_baseline_summary['train_r2']:.3f}"],
            [f"{final_baseline_summary['train_corr']:.3f}"],
            [f"{final_baseline_summary['residual_skew']:.3f}"],
        ],
        font=dict(size=10),
        align="left",
    ),
)])
fig.update_layout(title="Final MAEST Full+Peak30 PCA64 Ridge Fit", width=1450, height=250, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

print(
    f"Final fitted model variable: final_maest_full_peak30_pca64_ridge_model | "
    f"PCA/view={FINAL_MAEST_PCA_COMPONENTS}, alpha={FINAL_RIDGE_ALPHA:.4g}"
)


Final fitted model variable: final_maest_full_peak30_pca64_ridge_model | PCA/view=64, alpha=790.6


In [26]:
alpha_df = sorted(final_alpha_rows, key=lambda row: row["alpha"])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[row["alpha"] for row in alpha_df],
    y=[row["mae"] for row in alpha_df],
    mode="lines+markers",
))
fig.update_xaxes(type="log", title="Ridge alpha")
fig.update_yaxes(title="CV MAE")
fig.update_layout(title="Final Full+Peak30 Model Alpha Search Curve", height=420)
fig.show()

alpha_df

[{'alpha': 0.01, 'mae': 1.1601379572795292},
 {'alpha': 0.013257113655901088, 'mae': 1.1598670392695738},
 {'alpha': 0.017575106248547922, 'mae': 1.1595083200516234},
 {'alpha': 0.023299518105153717, 'mae': 1.159033533041981},
 {'alpha': 0.030888435964774818, 'mae': 1.158405453292706},
 {'alpha': 0.040949150623804255, 'mae': 1.1575751644453238},
 {'alpha': 0.054286754393238594, 'mae': 1.1564785711017282},
 {'alpha': 0.07196856730011521, 'mae': 1.1550320078965786},
 {'alpha': 0.09540954763499938, 'mae': 1.1531268144606626},
 {'alpha': 0.12648552168552957, 'mae': 1.1506228105886158},
 {'alpha': 0.16768329368110083, 'mae': 1.1473573129740093},
 {'alpha': 0.22229964825261944, 'mae': 1.1432154206673575},
 {'alpha': 0.29470517025518095, 'mae': 1.1379808758502477},
 {'alpha': 0.3906939937054617, 'mae': 1.1312926416533213},
 {'alpha': 0.517947467923121, 'mae': 1.1230856831869576},
 {'alpha': 0.6866488450043002, 'mae': 1.1126907287111412},
 {'alpha': 0.9102981779915218, 'mae': 1.099606281561975

In [31]:
raw_cv_mae = mean_absolute_error(y_final[valid_cal], raw_oof[valid_cal])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_final[valid_cal],
    y=raw_oof[valid_cal],
    mode="markers",
    name=f"OOF predictions | CV MAE = {raw_cv_mae:.3f}",
    marker=dict(
        color="#1f77b4",
        size=8,
        opacity=0.8,
        line=dict(width=0.5, color="black"),
    ),
    text=final_labels[valid_cal],
    hovertemplate="%{text}<br>actual=%{x:.2f}<br>raw pred=%{y:.2f}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=[1, 9],
    y=[1, 9],
    mode="lines",
    name="Perfect prediction",
    line=dict(color="black", dash="dash"),
))

fig.update_layout(
    title="MAEST Full+Peak30 PCA64 Ridge Fit: Predictions vs Tagged Energy",
    xaxis_title="Tagged energy",
    yaxis_title="OOF prediction",
    height=520,
    width=700,
    margin=dict(l=60, r=30, t=70, b=60),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)"),
)

fig.show()

## Isotonic Calibration Experiment

This section tests whether a monotonic calibration layer can correct residual skew in the PCA64 Ridge predictions. The evaluation is nested out-of-fold: each calibration curve is fit only from predictions made inside the training fold, then applied to that fold's held-out tracks.


In [27]:
from sklearn.isotonic import IsotonicRegression

CALIBRATION_OUTER_SPLITS = 5
CALIBRATION_INNER_SPLITS = 5

def tune_alpha_on_indices(X, y, train_idx, pipeline, alpha_grid, *, splits=3, random_state=42):
    inner_cv = KFold(n_splits=min(splits, len(train_idx)), shuffle=True, random_state=random_state)
    best_alpha = None
    best_mae = float("inf")
    for alpha in alpha_grid:
        fold_maes = []
        for inner_train_rel, inner_test_rel in inner_cv.split(X[train_idx], y[train_idx]):
            inner_train_idx = train_idx[inner_train_rel]
            inner_test_idx = train_idx[inner_test_rel]
            model = clone(pipeline)
            model.set_params(ridge__alpha=float(alpha))
            model.fit(X[inner_train_idx], y[inner_train_idx])
            pred = model.predict(X[inner_test_idx])
            fold_maes.append(mean_absolute_error(y[inner_test_idx], pred))
        alpha_mae = float(np.mean(fold_maes))
        if alpha_mae < best_mae:
            best_mae = alpha_mae
            best_alpha = float(alpha)
    return best_alpha

def training_oof_predictions(X, y, train_idx, pipeline, alpha, *, splits=CALIBRATION_INNER_SPLITS, random_state=42):
    train_oof = np.full(len(train_idx), np.nan, dtype=np.float64)
    inner_cv = KFold(n_splits=min(splits, len(train_idx)), shuffle=True, random_state=random_state)
    for inner_train_rel, inner_cal_rel in inner_cv.split(X[train_idx], y[train_idx]):
        inner_train_idx = train_idx[inner_train_rel]
        inner_cal_idx = train_idx[inner_cal_rel]
        model = clone(pipeline)
        model.set_params(ridge__alpha=float(alpha))
        model.fit(X[inner_train_idx], y[inner_train_idx])
        train_oof[inner_cal_rel] = model.predict(X[inner_cal_idx])
    return train_oof

def regression_metrics(label, y_true, pred):
    residual = y_true - pred
    return {
        "model": label,
        "mae": float(mean_absolute_error(y_true, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred))),
        "r2": float(r2_score(y_true, pred)),
        "corr": float(np.corrcoef(y_true, pred)[0, 1]) if np.std(pred) > 0 else float("nan"),
        "residual_mean": float(np.mean(residual)),
        "residual_std": float(np.std(residual)),
        "residual_skew": residual_skew(residual),
    }

calibration_pipeline = clone(final_base_pipeline)

outer_cv = KFold(n_splits=min(CALIBRATION_OUTER_SPLITS, len(y_final)), shuffle=True, random_state=123)
raw_oof = np.full_like(y_final, np.nan, dtype=np.float64)
calibrated_oof = np.full_like(y_final, np.nan, dtype=np.float64)
calibration_fold_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_final, y_final), start=1):
    alpha = tune_alpha_on_indices(
        X_final,
        y_final,
        train_idx,
        calibration_pipeline,
        FINAL_ALPHA_GRID,
        splits=INNER_ALPHA_SPLITS,
        random_state=500 + fold_idx,
    )
    train_raw_oof = training_oof_predictions(
        X_final,
        y_final,
        train_idx,
        calibration_pipeline,
        alpha,
        splits=CALIBRATION_INNER_SPLITS,
        random_state=700 + fold_idx,
    )
    train_valid = np.isfinite(train_raw_oof)
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(train_raw_oof[train_valid], y_final[train_idx][train_valid])

    model = clone(calibration_pipeline)
    model.set_params(ridge__alpha=float(alpha))
    model.fit(X_final[train_idx], y_final[train_idx])
    raw_pred = model.predict(X_final[test_idx])
    cal_pred = calibrator.predict(raw_pred)

    raw_oof[test_idx] = raw_pred
    calibrated_oof[test_idx] = cal_pred
    calibration_fold_rows.append({
        "fold": fold_idx,
        "alpha": alpha,
        "raw_mae": float(mean_absolute_error(y_final[test_idx], raw_pred)),
        "calibrated_mae": float(mean_absolute_error(y_final[test_idx], cal_pred)),
    })

valid_cal = np.isfinite(raw_oof) & np.isfinite(calibrated_oof)
calibration_metrics = [
    regression_metrics("raw_full_peak30_pca64_ridge_oof", y_final[valid_cal], raw_oof[valid_cal]),
    regression_metrics("isotonic_calibrated_oof", y_final[valid_cal], calibrated_oof[valid_cal]),
]

# Fit a final calibrator for future predictions: final model raw predictions are calibrated by a curve
# learned from OOF raw predictions on the labeled training set.
final_isotonic_calibrator = IsotonicRegression(out_of_bounds="clip")
final_isotonic_calibrator.fit(raw_oof[valid_cal], y_final[valid_cal])
final_calibrated_train_pred = final_isotonic_calibrator.predict(final_train_pred)

fig = go.Figure(data=[go.Table(
    columnwidth=[2.5, 0.8, 0.8, 0.8, 0.8, 1.0, 1.0, 1.0],
    header=dict(
        values=["Model", "OOF MAE", "OOF RMSE", "OOF R2", "OOF corr", "Residual mean", "Residual std", "Residual skew"],
        fill_color="#17202a",
        font=dict(color="white", size=11),
        align="left",
    ),
    cells=dict(
        values=[
            [row["model"] for row in calibration_metrics],
            [f"{row['mae']:.3f}" for row in calibration_metrics],
            [f"{row['rmse']:.3f}" for row in calibration_metrics],
            [f"{row['r2']:.3f}" for row in calibration_metrics],
            [f"{row['corr']:.3f}" for row in calibration_metrics],
            [f"{row['residual_mean']:+.3f}" for row in calibration_metrics],
            [f"{row['residual_std']:.3f}" for row in calibration_metrics],
            [f"{row['residual_skew']:+.3f}" for row in calibration_metrics],
        ],
        font=dict(size=10),
        align="left",
    ),
)])
fig.update_layout(title="OOF Isotonic Calibration Check", width=1150, height=260, margin=dict(l=20, r=20, t=60, b=20))
fig.show()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Raw full+peak30 PCA64 Ridge", "Isotonic calibrated"))
for col, pred, title in [(1, raw_oof, "raw"), (2, calibrated_oof, "calibrated")]:
    fig.add_trace(go.Scatter(
        x=y_final[valid_cal],
        y=pred[valid_cal],
        mode="markers",
        marker=dict(color=y_final[valid_cal], colorscale="Turbo", size=8, line=dict(width=0.5, color="black")),
        text=final_labels[valid_cal],
        hovertemplate="%{text}<br>actual=%{x:.2f}<br>pred=%{y:.2f}<extra></extra>",
        showlegend=False,
    ), row=1, col=col)
    fig.add_shape(type="line", x0=1, y0=1, x1=9, y1=9, line=dict(color="black", dash="dash"), row=1, col=col)
    fig.update_xaxes(title_text="Tagged energy", row=1, col=col)
    fig.update_yaxes(title_text=f"{title} OOF prediction", row=1, col=col)
fig.update_layout(title="Raw vs Isotonic-Calibrated OOF Predictions", height=520, margin=dict(l=50, r=20, t=80, b=50))
fig.show()

raw_mae = calibration_metrics[0]["mae"]
cal_mae = calibration_metrics[1]["mae"]
print(
    f"Calibration MAE delta: {cal_mae - raw_mae:+.3f} "
    f"(negative means isotonic improved held-out MAE)."
)
print("Final calibration variables: final_maest_full_peak30_pca64_ridge_model, final_isotonic_calibrator")


Calibration MAE delta: +0.044 (negative means isotonic improved held-out MAE).
Final calibration variables: final_maest_full_peak30_pca64_ridge_model, final_isotonic_calibrator
